# Expfit notes 4: Confidence

\begin{align}
E(\theta) = \frac{1}{n}\sum(v_i - f(\theta)_i)^2
\end{align}

## Gaussian log likelihood

\begin{align}
L(\theta, \sigma | v) \equiv p(v | \theta, \sigma)
 &= \prod \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left( \frac{(v_i - f(\theta)_i)^2}{2\sigma^2} \right)
\end{align}

\begin{align}
\mathcal{l}(\theta, \sigma | v) \equiv \log L(\theta, \sigma | v)
 &= -\frac{n}{2}\log(2\pi) - n\log(\sigma) - \frac{1}{2\sigma^2} \sum (v_i - f(\theta)_i)^2
\end{align}

\begin{align}
\frac{\partial}{\partial \theta} \mathcal{l}(\theta | \sigma, v)
 &=  -\frac{1}{2\sigma^2} \frac{\partial}{\partial \theta} \sum (v_i - f(\theta)_i)^2
\end{align}

## Fisher information matrix

**Asymptotic normality**: regardless of the exact likelihood, near the true solution $\theta_\text{true}$, the _maximum likelihood estimate_ $\hat{\theta}$ has a Normal distribution:
\begin{align}
\hat{\theta} \sim N(\theta_\text{true}, I(\theta_\text{true})^{-1})
\end{align}
where $\hat{\theta}$ is a random variable because it's based on randomly sampled data.

The quantity $I(\theta)$ is called the Fisher information matrix, and is defined as
\begin{align}
I_{ij}(\theta) = E \left( -\frac{\partial^2\mathcal{l}(\theta|v)}{\partial \theta_i \partial \theta_j} \right)
\end{align}
but approximated by
\begin{align}
J_{ij}(\hat{\theta}) = \left. -\frac{\partial^2\mathcal{l}(\theta|v)}{\partial \theta_i \partial \theta_j} \right\vert_{\theta = \hat{\theta}}
\end{align}

So we can approximate the covariance matrix by
\begin{align}
\Sigma = J^{-1}
\end{align}

## Gaussian log likelihood and MSE

The mean-squared error

\begin{align}
E(\theta) &= \frac{1}{n} \sum \left(v_i - f(\theta)_i \right)^2
\end{align}

relates to a Gaussian log likelihood

\begin{align}
\mathcal{l}(\theta, \sigma | v) \equiv \log L(\theta, \sigma | v)
 &= -\frac{n}{2}\log(2\pi) - n\log(\sigma) - \frac{1}{2\sigma^2} \sum (v_i - f(\theta)_i)^2
\end{align}

as

\begin{align}
n E(\theta) = -2\sigma^2(\mathcal{l}(\theta) + nc)
\end{align}

### Fixed, known, $\sigma$ and $n$

Taking the second derivative w.r.t. $\theta$ of each side (treating $\sigma$ and $n$ as known), we find

\begin{align}
n H(\theta) = -2\sigma^2 \frac{\partial^2}{\partial \theta^2}\mathcal{l}(\theta)
\end{align}
for
\begin{align}
J(\theta) = -\frac{\partial^2}{\partial \theta^2}\mathcal{l}(\theta) = \frac{n}{2\sigma^2} H(\theta)
\end{align}

So it seems that

- To use a Gaussian log likelihood to set confidence bounds, we need to also estimate $\sigma$
- We can't get nice confidence bounds without assuming a particular noise model

The second point particularly makes this a less interesting route.

### Estimated $\sigma$ ?

What if we estimate $\sigma$ from the residuals?
This means filling in
\begin{align}
\sigma^2 \approx \frac{1}{n} \sum (v_i - f(\theta)_i)^2 = E(\theta)
\end{align}

For 
\begin{align}
J(\theta) \approx \frac{n}{2}\frac{H(\theta)}{E(\theta)}
\end{align}
and
\begin{align}
\Sigma(\theta) \approx \frac{2E(\theta)}{n} H^{-1}(\theta)
\end{align}

Note that $n$ appears in $E$ and $H$, so $E$ adds a term $1/n$, which is cancelled out by $H^{-1}$ 

## Profile likelihood ratio confidence interval

Here, we fix the parameter of interest and re-run the optimisation, then compare the result with the original (best) result.

We can use a [likelihood ratio test](https://en.wikipedia.org/wiki/Likelihood-ratio_test) to compare

\begin{align}
-2 \log\left(\frac{L_\text{test}}{L_\text{best}}\right) < \chi^2_{1,1-\alpha}
\end{align}

where $\chi_{1,1-\alpha}$ is the value [Chi-squared distribution](https://en.wikipedia.org/wiki/Chi-squared_distribution) with $1$ degree of freedom and $1-\alpha$ is the level of confidence (i.e. for 5% we get $\chi_{1,0.95}$).

### Getting the test statistic

We can look it up in a table, or ask Scipy:

In [2]:
import scipy
print(scipy.stats.chi2.ppf(0.99, 1))
print(scipy.stats.chi2.ppf(0.95, 1))
print(scipy.stats.chi2.ppf(0.90, 1))

6.6348966010212145
3.841458820694124
2.705543454095404


In [4]:
import scipy
print(scipy.stats.chi2.ppf(0.01, 1))
print(scipy.stats.chi2.ppf(0.05, 1))
print(scipy.stats.chi2.ppf(0.10, 1))

0.00015708785790970184
0.003932140000019522
0.01579077409343122


### Comparing with MSE, assuming Gaussian noise

We can rewrite as log likelihoods

\begin{align}
-2 \log\left(\frac{L_\text{test}}{L_\text{best}}\right) = -2 (\mathcal{l}_\text{test} - \mathcal{l}_\text{best})
\end{align}

and then, because $n$ and $\sigma$ are the same, we get
\begin{align}
-2 \cdot \frac{-1}{2\sigma^2} \left(\sum (v_i - f(\theta_\text{test})_i)^2 - \sum (v_i - f(\theta_\text{best})_i)^2\right)
= \frac{n}{\sigma^2} (E_\text{test} - E_\text{best})
\end{align}

Using the same trick of equating $E_\text{best}$ with $\sigma^2$, we get

\begin{align}
\frac{n}{\sigma^2} (E_\text{test} - E_\text{best}) \approx n \left(\frac{E_\text{test}}{E_\text{best}} - 1\right) < \chi^2_{1,1-\alpha}
\end{align}

which would mean our confidence interval includes all values for which

\begin{align}
E_\text{test} < (1 + \chi^2_{1,1-\alpha} / n) E_\text{best}
\end{align}

so if $\chi^2$ is chosen larger, the upper bound for $E_\text{test}$ goes up?

And if $n$ is larger, the region shrinks

